In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce;

In [0]:
# COMMAND ----------
# MAGIC %sql
# MAGIC CREATE TABLE IF NOT EXISTS ecommerce.source_metadata (
# MAGIC     table_name STRING
# MAGIC );
# MAGIC 
# MAGIC INSERT INTO ecommerce.source_metadata (table_name) VALUES 
# MAGIC ('users'),
# MAGIC ('sellers'),
# MAGIC ('buyers'),
# MAGIC ('countries');

# COMMAND ----------
df = spark.table("ecommerce.source_metadata")
table_list = [row.table_name for row in df.collect()]

for table_name in ["users", "sellers", "buyers", "countries"]:
    bronze_table = f"ecommerce.{table_name}_raw"
    adls_path = f"abfss://landing@saecommercedataprod001.dfs.core.windows.net/{table_name}/"
    
    # Drop the existing table if it's causing a format conflict
    spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
    
    # Read parquet and write fresh
    df = spark.read.parquet(adls_path)
    df.write.mode("overwrite").saveAsTable(bronze_table)
    print(f"Successfully populated {bronze_table}")

Successfully populated ecommerce.users_raw
Successfully populated ecommerce.sellers_raw
Successfully populated ecommerce.buyers_raw
Successfully populated ecommerce.countries_raw


In [0]:
# Read the parquet files and overwrite/create the table properly
df = spark.read.parquet("abfss://landing@saecommercedataprod001.dfs.core.windows.net/users/")
df.write.mode("overwrite").saveAsTable("ecommerce.users_raw")

In [0]:
df = spark.read.parquet("abfss://landing@saecommercedataprod001.dfs.core.windows.net/users/")
df.printSchema()
display(df.limit(5))

root
 |-- identifierHash: double (nullable = true)
 |-- type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- language: string (nullable = true)
 |-- socialNbFollowers: integer (nullable = true)
 |-- socialNbFollows: integer (nullable = true)
 |-- socialProductsLiked: integer (nullable = true)
 |-- productsListed: integer (nullable = true)
 |-- productsSold: integer (nullable = true)
 |-- productsPassRate: integer (nullable = true)
 |-- productsWished: integer (nullable = true)
 |-- productsBought: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- civilityGenderId: integer (nullable = true)
 |-- civilityTitle: string (nullable = true)
 |-- hasAnyApp: string (nullable = true)
 |-- hasAndroidApp: string (nullable = true)
 |-- hasIosApp: string (nullable = true)
 |-- hasProfilePicture: string (nullable = true)
 |-- daysSinceLastLogin: integer (nullable = true)
 |-- seniority: integer (nullable = true)
 |-- seniorityAsMonths: double (nullable = true)

identifierHash,type,country,language,socialNbFollowers,socialNbFollows,socialProductsLiked,productsListed,productsSold,productsPassRate,productsWished,productsBought,gender,civilityGenderId,civilityTitle,hasAnyApp,hasAndroidApp,hasIosApp,hasProfilePicture,daysSinceLastLogin,seniority,seniorityAsMonths,seniorityAsYears,countryCode
-7.27964E18,user,Etats-Unis,en,3,8,0,0,0,0,0,0,F,2,mrs,FALSE,FALSE,FALSE,TRUE,709,3205,106.83,8.9,us
-1.45601E18,user,Allemagne,de,3,8,0,0,0,0,0,0,F,2,mrs,FALSE,FALSE,FALSE,TRUE,709,3205,106.83,8.9,de
9.00628E18,user,SuÃ¨de,en,3,8,0,0,0,0,0,0,M,1,mr,TRUE,FALSE,TRUE,TRUE,689,3205,106.83,8.9,se
-7.15463E18,user,Turquie,en,3,8,0,0,0,0,0,0,F,2,mrs,FALSE,FALSE,FALSE,TRUE,709,3205,106.83,8.9,tr
2.8583E18,user,France,en,3,8,0,0,0,0,0,0,M,1,mr,TRUE,FALSE,TRUE,TRUE,709,3205,106.83,8.9,fr
